<a href="https://colab.research.google.com/github/NancyWangareh/ReCover/blob/main/ReCover_Information_and_RecommendationSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReCover: Insurance Information and Recommendation Assistant for Kenyan Traders

This notebook builds **one chatbot with two jobs** for market traders (for example mitumba sellers), in English, Kiswahili or Sheng:
1. **Information:** answers insurance questions only from the approved documents, shows where each fact came from, and says so when the documents don't have the answer.
2. **Recommendation:** asks a few questions about the trader's business, sends the answers to the team's recommendation service, and explains the suggested plans in plain language.

## How the project requirements are met
| Requirement | Where in this notebook |
|---|---|
| Use an LLM to interpret questions in everyday language | Section 8 (`make_queries`) rewrites questions for searching; Section 9 (`interpret_message`) reads the trader's facts, language and intent |
| Retrieve relevant information from the approved documents | Sections 3-7 read the PDFs, cut them into chunks and store them in a searchable library; Section 8 (`retrieve`) searches it |
| Apply rules for eligibility, coverage, premiums, exclusions | Section 10: plain Python rules, no AI involved |
| Return source references | Every fact is labelled `[S1]` (document page), `[R1]` (rule) or `[A1]` (recommendation service); the answer ends with a source list |
| Say when information is unavailable | `retrieve` drops weak matches; `ReCover.inform` returns a fixed message when nothing relevant is found; the instructions forbid guessing |
| Recommend products | Section 12 collects the details the recommendation service needs and calls it; Section 13 routes each message to information or recommendation |



## 0. Install the libraries we use
- `pypdf` reads PDFs
- `langchain-text-splitters` cuts long text into chunks
- `chromadb` stores the searchable library
- `sentence-transformers` turns text into numbers (embeddings) so we can search by meaning
- `google-genai` talks to Gemini
- `pandas` + `openpyxl` read the Excel workbook

In [ ]:
%pip install -q pypdf langchain-text-splitters chromadb sentence-transformers google-genai pandas openpyxl requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 1. Settings and getting the repo's data
Colab starts as an empty computer every session, so we first copy (clone) the team repo into it. All the approved documents live in the repo's `data collection` folder.

All the settings you may want to change are in one place below.

In [ ]:
import json
import os
import re
import subprocess
from pathlib import Path

import pandas as pd


REPO_URL = "https://github.com/NancyWangareh/ReCover.git"
MODEL = "gemini-3.8-flash"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "recover_docs"
MIN_SIMILARITY = 0.30
TOP_K = 4
MAX_RESULTS = 5
RECOMMENDER_URL = "https://recover-rwf9.onrender.com/recommend"
RECOMMENDER_TIMEOUT = 90


def find_docs_dir():
    """Look for the folder holding the approved documents."""
    for candidate in [Path("ReCover/data collection"), Path("data collection")]:
        if candidate.exists():
            return candidate
    return None



if find_docs_dir() is None:
    if "<owner>" in REPO_URL:
        raise ValueError("Edit REPO_URL above and put in your team's repo link, then run this cell again.")
    subprocess.run(["git", "clone", REPO_URL], check=True)

DOCS_DIR = find_docs_dir()
if DOCS_DIR is None:
    raise FileNotFoundError("Could not find the 'data collection' folder after cloning. Check REPO_URL.")
REPO_ROOT = DOCS_DIR.parent
print("Documents folder:", DOCS_DIR)


def get_api_key():
    """Read the Gemini key from Colab Secrets, or from an environment variable when running locally.
    The key is never written in the notebook, which matters because this notebook goes on GitHub."""
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        key = os.environ.get("GEMINI_API_KEY")
        if not key:
            raise RuntimeError("No Gemini key found. Add GEMINI_API_KEY to Colab Secrets or set it as an environment variable.")
        return key

Documents folder: ReCover/data collection


## 2. Look at the data we have
Before building anything, list every file in the repo and check that the files this notebook relies on are present.

In [ ]:

for f in sorted(REPO_ROOT.rglob("*")):
    if ".git" not in f.parts and f.is_file():
        print(f"- {f.relative_to(REPO_ROOT)} ({f.stat().st_size:,} bytes)")


needed = [
    "Microinsurance_Data_Collection_Workbook.xlsx",
    "ReCover_Microinsurance_Product_Matching_Database.xlsx",
]
print()
for name in needed:
    found = list(REPO_ROOT.rglob(name))
    print(("FOUND   " if found else "MISSING ") + name)

- .gitignore (466 bytes)
- Information_system.ipynb (543,409 bytes)
- LICENSE (1,070 bytes)
- README.md (11,647 bytes)
- ReCover_Chatbot_.ipynb (270,751 bytes)
- ReCover_Trader_Analysis.ipynb (2,524,825 bytes)
- data/README.md (1 bytes)
- data/ReCover_Microinsurance_Product_Matching_Database.xlsx (16,239 bytes)
- data/traders.csv (88,447 bytes)
- data collection/BRITAM-PRODUCTS-PROFILE-Final-Final.pdf (13,301,344 bytes)
- data collection/Microinsurance_Data_Collection_Workbook.xlsx (29,634 bytes)
- data collection/TCIP-BROCHURE.pdf (677,726 bytes)
- data collection/THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf (323,455 bytes)
- data collection/jinue_policy_document.pdf (312,266 bytes)
- data collection/ke-act-ln-2020-26-publication-document.pdf (84,865 bytes)
- docs/README.md (1 bytes)
- main.py (339 bytes)
- models/recover_model_bundle.pkl (10,115 bytes)
- python-version (8 bytes)
- recommendation_api/main.py (2,171 bytes)
- recommendation_api/recommender.py (11,190 bytes)
-

## 3. Check which PDFs the computer can actually read
Some PDFs contain real text. Others are **scans**: every page is a photo of text, and code can't read a photo. We test each PDF by asking for its text and counting the characters. Files with no text need the extra OCR step in Section 4.

In [ ]:
from pypdf import PdfReader

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
needs_ocr = []   # PDFs where pypdf found no text at all

for pdf in pdf_files:
    reader = PdfReader(str(pdf))
    # extract_text() returns nothing for image-only pages, so "or ''" avoids a crash
    chars = sum(len(page.extract_text() or "") for page in reader.pages)
    print(f"{pdf.name}: {len(reader.pages)} pages, {chars:,} characters of text")
    if chars == 0:
        needs_ocr.append(pdf.name)

print("\nScanned PDFs that need OCR:", needs_ocr)

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 20 pages, 0 characters of text
TCIP-BROCHURE.pdf: 2 pages, 0 characters of text
THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf: 51 pages, 91,854 characters of text
jinue_policy_document.pdf: 18 pages, 61,684 characters of text
ke-act-ln-2020-26-publication-document.pdf: 7 pages, 16,327 characters of text

Scanned PDFs that need OCR: ['BRITAM-PRODUCTS-PROFILE-Final-Final.pdf', 'TCIP-BROCHURE.pdf']


## 4. Read the scanned PDFs with Gemini (OCR)
OCR means turning a picture of text into real text. We send each scanned PDF to Gemini and ask for a word-for-word copy, with a `=== PAGE n ===` line before each page so every fact can later be traced to its page.

Two safeguards:
- **temperature=0** asks for the most literal output (no creativity).
- **Results are saved** to `data/ocr_text/`, so re-running the notebook doesn't pay for OCR twice. Delete the `.txt` file if you want a fresh transcription.

**Always spot-check the result** (next cell). A wrong number in the transcription becomes a wrong number in the chatbot.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())

OCR_DIR = Path("data/ocr_text")
OCR_DIR.mkdir(parents=True, exist_ok=True)

OCR_PROMPT = """Transcribe ALL text in this PDF exactly as printed.
Do not summarize, translate, or correct anything, and keep every number exactly as written.
Start each page with a line like: === PAGE 1 === (number = the page's position in the PDF, starting at 1).
If a page has no text, write [no text] under its page line."""

for name in needs_ocr:
    out_file = OCR_DIR / f"{Path(name).stem}.txt"
    if out_file.exists():
        print(f"{name}: already transcribed, skipping")
        continue
    uploaded = client.files.upload(file=str(DOCS_DIR / name))   # send the PDF to Gemini
    resp = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, OCR_PROMPT],
        config=types.GenerateContentConfig(temperature=0),
    )
    out_file.write_text(resp.text, encoding="utf-8")
    print(f"{name}: {len(resp.text):,} characters saved")

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 23,901 characters saved
TCIP-BROCHURE.pdf: 6,337 characters saved


### Spot-check the transcriptions
Compare what prints here with the actual PDF pages. For the current documents these are the facts to look for:
- **Britam guide:** premium rates such as `0.175%` (fire), `0.55%` (theft), `5%` (money), `3%` (transit), `0.1%` (public liability). The All Risks rate is printed `15% (1.5%)` in the original, so a transcription that copies it as printed is correct.
- **Jubilee TCIP brochure:** the KES 15,000,000 limit, the wall and roof rule, and the excluded businesses.

If you add other scanned documents, check their key numbers the same way.

In [ ]:
for name in needs_ocr:
    text = (OCR_DIR / f"{Path(name).stem}.txt").read_text(encoding="utf-8")
    print(f"\n===== {name} =====")
    for line in text.splitlines():
        # show lines containing a percentage, the stock limit, or exclusion/premises wording
        if any(key in line.lower() for key in ["%", "15,000,000", "exclud", "walls"]):
            print(line)


===== BRITAM-PRODUCTS-PROFILE-Final-Final.pdf =====
• Premium rate of 0.175% (1.75‰)
• Premium rate on full cover is 0.55%
• Premium Rate on Full cover of 15% (1.5%)
• Premium Rate is 5%
• Premium rate is 3%
• A simplified rating of 3% of the wage-roll
• Premium rate is 0.1%
The NSSF Act 2013 was introduced on 24 December 2013 to replace the NSSF Act Cap 258 of 1965. It was not fully implemented until 3 February 2023 when the Court of Appeal declared the Act constitutional and directed its implementation with immediate effect. The Act requires that both you and your employees increase your NSSF contributions from KES 200 per month to 6% of pensionable salary (subject to certain salary maximums). These contributions will be on a graduated scale referred to as tiers.
These are calculated as 6% of an employee’s pensionable income up to a maximum referred to as the Lower Earnings Limit (L.E.L). This is currently capped at KES 7,000 per month. The maximum contribution for Tier 1 contributi

## 5. Put every document into one list of pages
Each page becomes a small record with three labels: which file (`source`), which page (`page`), and its `text`. Keeping the page number is what lets the bot say "TCIP-BROCHURE.pdf, p.1" later.

- Readable PDFs: the text comes straight from `pypdf`.
- Scanned PDFs: the text comes from our saved OCR file, split at the `=== PAGE n ===` lines.

In [ ]:
def split_ocr_pages(raw):
    """Cut an OCR text file into (page number, text) pairs using the === PAGE n === markers."""
    parts = re.split(r"=== PAGE (\d+) ===", raw)
    # re.split gives: [text before first marker, "1", text of page 1, "2", text of page 2, ...]
    return [(int(num), text.strip()) for num, text in zip(parts[1::2], parts[2::2])]


pages = []   # one entry per page: {"source": file name, "page": number, "text": text}

for pdf in pdf_files:
    if pdf.name in needs_ocr:
        raw = (OCR_DIR / f"{pdf.stem}.txt").read_text(encoding="utf-8")
        ocr_pages = split_ocr_pages(raw)
        expected = len(PdfReader(str(pdf)).pages)
        if len(ocr_pages) != expected:
            # A mismatch means a page marker is missing or duplicated, so page citations could be wrong.
            print(f"WARNING: {pdf.name} has {expected} pages but the transcription has {len(ocr_pages)}. "
                  "Delete its .txt file in data/ocr_text and re-run Section 4.")
        for num, text in ocr_pages:
            pages.append({"source": pdf.name, "page": num, "text": text})
    else:
        reader = PdfReader(str(pdf))
        for i, page in enumerate(reader.pages, start=1):   # start=1 because humans count pages from 1
            pages.append({"source": pdf.name, "page": i, "text": (page.extract_text() or "").strip()})

# How many pages did each file contribute? (should equal each PDF's page count)
print(pd.Series([p["source"] for p in pages]).value_counts().to_string())

THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf    51
BRITAM-PRODUCTS-PROFILE-Final-Final.pdf                    20
jinue_policy_document.pdf                                  18
ke-act-ln-2020-26-publication-document.pdf                  7
TCIP-BROCHURE.pdf                                           2


## 6. Cut the pages into chunks
A whole page is too big to search well: it may cover several topics, and only one paragraph answers the question. We cut each page into pieces of about 1,000 characters.

- `chunk_overlap=150`: neighbouring chunks share 150 characters, so a sentence cut at the boundary doesn't lose its meaning.
- `separators`: the splitter prefers to cut at blank lines, then new lines, then sentence ends, and only cuts mid-word as a last resort.
- Each chunk keeps its `source` and `page` for citations.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for p in pages:
    if not p["text"] or p["text"] == "[no text]":
        continue   # skip empty pages (for example photo-only pages)
    for i, piece in enumerate(splitter.split_text(p["text"])):
        if len(piece.strip()) >= 30:   # drop tiny fragments such as a lone page number
            chunks.append({
                "id": f"{p['source']}-p{p['page']}-c{i}",   # unique name for this chunk
                "source": p["source"],
                "page": p["page"],
                "text": piece.strip(),
            })

print(f"{len(chunks)} chunks")
print("\nExample chunk:", chunks[0]["id"])
print(chunks[0]["text"][:300])

274 chunks

Example chunk: BRITAM-PRODUCTS-PROFILE-Final-Final.pdf-p1-c0
Britam
With you every step of the way

Britam Biashara Network
Product Guide
—
Insurance and Investment Solutions for
business protection and growth.


## 7. Build the searchable library
An **embedding model** turns a piece of text into a list of 384 numbers. Texts with similar *meaning* get similar numbers, so "how much does theft cover cost?" lands near the paragraph about the theft premium even if the words differ.

We store these numbers in **ChromaDB**, a small database that finds the closest matches quickly. The first run downloads the embedding model, so it takes a few minutes.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)

# Convert every chunk's text into numbers.
# normalize_embeddings=True scales each list to the same length, so closeness depends on meaning, not chunk length.
vectors = embedder.encode(
    [c["text"] for c in chunks],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

Path("data/chroma_db").mkdir(parents=True, exist_ok=True)
chroma = chromadb.PersistentClient(path="data/chroma_db")   # saved on disk

# Delete any old copy first, so re-running this cell never crashes on duplicate ids or keeps stale chunks.
if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(COLLECTION_NAME)
collection = chroma.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})   # cosine = standard closeness measure for text

collection.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=vectors,
    metadatas=[{"source": c["source"], "page": c["page"]} for c in chunks],   # this is what lets us cite pages later
)
print(f"Stored {collection.count()} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Stored 274 chunks


## 8. Finding the right pages for a question

**Why we rewrite questions first.** A trader may ask "which businesses are excluded from the Jubilee policy?" but you have two different Jubilee documents (TCIP for businesses and J-Inue for health). Searched as written, the correct page ranked 37th and never reached the chatbot. When Gemini rewrites the question with the full product names, it ranks 2nd.

**Why we also keep the original question.** When we used *only* the rewrites, the theft-premium question got worse: the long product names matched title pages better than the theft paragraph. So the original question is always searched too, and the rewrites are a bonus.

`retrieve()` runs these searches, merges duplicates, drops chunks below `MIN_SIMILARITY`, and returns the best few.

In [ ]:
# A short description of each document, so Gemini knows which products exist.
# Edit the legal-notice line if its description is wrong (I only know it by its file name).
DOCS_GUIDE = """Documents available:
- Britam Biashara Network Product Guide: business insurance (fire, theft/burglary, money, all risks, goods in transit, work injury, public liability), plus group medical, funeral, group life
- Jubilee Traders Comprehensive Insurance Policy (TCIP) brochure: one-package business insurance for shops and traders
- Jubilee J-Inue policy document: health insurance
- Kenya Microinsurance Policy Paper (final draft)
- Kenya legal notice (file ke-act-ln-2020-26)"""

REWRITE_PROMPT = f"""You prepare search queries for a retrieval system about Kenyan insurance.
{DOCS_GUIDE}

Rewrite the trader's message as 1 to 2 standalone English search queries.
- KEEP the topic words from the message (for example 'theft', 'premium rate', 'excluded'). Do not replace them with product names.
- Add a product's full name only when the message clearly names that product, or when it is ambiguous which Jubilee or Britam product is meant.
- Never answer the question. Only write the queries.
Return a JSON list of strings."""


def make_queries(question):
    """Return the list of search queries: the trader's own question first, then up to 2 rewrites."""
    rewrites = []
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{REWRITE_PROMPT}\n\nTrader message: {question}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        parsed = json.loads(resp.text)
        if isinstance(parsed, list):   # only trust the output if it is really a list of strings
            rewrites = [q for q in parsed if isinstance(q, str)][:2]
    except Exception as e:
        print(f"(rewrite failed, using the original question only: {e})")
    return [question] + [q for q in rewrites if q != question]


def retrieve(question):
    """Find the chunks most relevant to the question. Returns [] when nothing is relevant enough."""
    best = {}   # chunk id -> best hit seen so far (a chunk may be found by more than one query)

    for query in make_queries(question):
        vec = embedder.encode(query, normalize_embeddings=True).tolist()
        res = collection.query(query_embeddings=[vec], n_results=TOP_K,
                               include=["documents", "metadatas", "distances"])
        for cid, doc, meta, dist in zip(res["ids"][0], res["documents"][0],
                                        res["metadatas"][0], res["distances"][0]):
            sim = 1 - dist   # the database gives distance; similarity = 1 - distance (higher = closer)
            if sim >= MIN_SIMILARITY and (cid not in best or sim > best[cid]["similarity"]):
                best[cid] = {"source": meta["source"], "page": meta["page"],
                             "text": doc, "similarity": sim}

    hits = sorted(best.values(), key=lambda h: h["similarity"], reverse=True)   # best matches first
    return hits[:MAX_RESULTS]


# Try it: the football question should find nothing, which is how the bot knows to say "I don't have that".
for q in ["What is the premium rate for theft cover?", "Who will win the football match on Sunday?"]:
    print("\nQUESTION:", q)
    found = retrieve(q)
    if not found:
        print("  -> nothing relevant found")
    for h in found:
        print(f"  {h['source']}, p.{h['page']} ({h['similarity']:.2f})")


QUESTION: What is the premium rate for theft cover?
  BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8 (0.62)
  TCIP-BROCHURE.pdf, p.1 (0.48)
  jinue_policy_document.pdf, p.11 (0.47)
  THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf, p.27 (0.44)
  ke-act-ln-2020-26-publication-document.pdf, p.2 (0.44)

QUESTION: Who will win the football match on Sunday?
  -> nothing relevant found


## 9. Read what the trader tells us
For every message, Gemini returns three things:
- the **facts** the trader stated (premises type, stock value, budget, income...). These go into a form that we fill in as the conversation goes on.
- the **language** (`en`, `sw` or `sheng`), so the reply is in the trader's language.
- the **intent**: does the trader want a product *recommendation*, or an *information* answer? This is how Section 13 decides which path to follow.

Rules for the form:
- Anything the trader didn't clearly say stays empty (`null`). We never guess.
- The budget amount and its period (day / week / month) are kept **separate**. Python converts them later.
- `clean_profile` doesn't trust Gemini's output blindly. It keeps only known fields with the right type and allowed values.
- The values allowed for `income_variability`, `income_receipt_pattern`, `preferred_premium_frequency` and `primary_risk_concern` are the labels I believe the recommendation model was trained on. **Check them against the team's trader dataset** and edit the sets below if they differ.

In [ ]:
PROFILE_PROMPT = """You read a message from a Kenyan trader talking to an insurance assistant and extract only what they stated.
Messages may be English, Kiswahili or Sheng. Return a JSON object with ONLY these keys:
- language: "en", "sw" (Kiswahili) or "sheng" (mixed Swahili/English/Sheng)
- intent: "recommendation" if the trader asks which plan or product to buy, asks for suggestions or options for their business, or is answering questions about their business and money; otherwise "information"
- stall_type: "open_air" (open-air stall, tarpaulin, no walls), "temporary_structure" (wooden or mabati kiosk), "permanent_building" (stone or brick shop, 'duka ya mawe'), or null
- business_type: short English text such as "mitumba shoes", or null
- stock_value_kes: whole number, or null
- budget_amount_kes: the amount the trader says they can pay for insurance, as a whole number, or null
- budget_period: "day", "week" or "month" (the period for that amount), or null
- wants_cover_fire, wants_cover_theft, wants_cover_health, wants_cover_accident: true if the trader says they want it or worry about it, otherwise null
- dependents: whole number, or null
- monthly_income_kes: what the trader earns in a month, whole number, or null
- income_receipt_pattern: how often money comes in: "daily", "weekly" or "monthly", or null
- income_variability: "low" (steady), "medium" (changes sometimes) or "high" (changes a lot), or null
- restock_cycle_days: whole number of days between restocking, or null
- total_loss_24m_kes: money or stock lost to fire, theft or similar in the last 2 years, whole number (0 if the trader says they lost nothing), or null if not mentioned
- preferred_premium_frequency: how they prefer to pay: "daily", "weekly", "monthly" or "pay_at_restock" (pays when they restock), or null
- primary_risk_concern: the one risk worrying them most: "fire", "theft", "health", "weather_flood", "accident" or "other", or null
Use null for anything not clearly stated. Never guess."""

LANGUAGES = {"en", "sw", "sheng"}
INTENTS = {"recommendation", "information"}
STALL_TYPES = {"open_air", "temporary_structure", "permanent_building"}
PERIODS = {"day", "week", "month"}
WANTS = ["wants_cover_fire", "wants_cover_theft", "wants_cover_health", "wants_cover_accident"]
INT_FIELDS = ["stock_value_kes", "budget_amount_kes", "dependents", "monthly_income_kes",
              "restock_cycle_days", "total_loss_24m_kes"]
# Allowed words for the "choice" fields. Check these against the team's trader dataset.
CHOICES = {
    "income_receipt_pattern": {"daily", "weekly", "monthly"},
    "income_variability": {"low", "medium", "high"},
    "preferred_premium_frequency": {"daily", "weekly", "monthly", "pay_at_restock"},
    "primary_risk_concern": {"fire", "theft", "health", "weather_flood", "accident", "other"},
}


def clean_profile(raw):
    """Keep only the fields we know, with valid values. Anything else from Gemini is thrown away."""
    out = {}
    if raw.get("stall_type") in STALL_TYPES:
        out["stall_type"] = raw["stall_type"]
    if isinstance(raw.get("business_type"), str) and raw["business_type"].strip():
        out["business_type"] = raw["business_type"].strip()
    for key in INT_FIELDS:
        v = raw.get(key)
        # isinstance(v, bool) check: Python treats True as the number 1, which we don't want
        if isinstance(v, (int, float)) and not isinstance(v, bool) and v >= 0:
            out[key] = int(v)
    if raw.get("budget_period") in PERIODS:
        out["budget_period"] = raw["budget_period"]
    for key, allowed in CHOICES.items():
        if raw.get(key) in allowed:
            out[key] = raw[key]
    for key in WANTS:
        if raw.get(key) is True:   # we only store a "want" when it is clearly True; missing means "not mentioned"
            out[key] = True
    return out


def interpret_message(message):
    """Ask Gemini to read the message. Returns {"profile": {...}, "language": ..., "intent": ...}.
    If anything goes wrong we fall back to: no new facts, English, information."""
    fallback = {"profile": {}, "language": "en", "intent": "information"}
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{PROFILE_PROMPT}\n\nTrader message: {message}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        raw = json.loads(resp.text)
        if not isinstance(raw, dict):
            return fallback
        return {
            "profile": clean_profile(raw),
            "language": raw["language"] if raw.get("language") in LANGUAGES else "en",
            "intent": raw["intent"] if raw.get("intent") in INTENTS else "information",
        }
    except Exception as e:
        print(f"(could not read the message: {e})")
        return fallback


for t in ["Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?",
          "Nataka ushauri wa bima ya moto. Stock yangu ni 500,000 bob, naweza lipa 1000 kwa mwezi.",
          "What is microinsurance?"]:
    print(t, "\n  ->", interpret_message(t), "\n")

Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali? 
  -> {'profile': {'stall_type': 'permanent_building'}, 'language': 'sw', 'intent': 'information'} 

Nataka ushauri wa bima ya moto. Stock yangu ni 500,000 bob, naweza lipa 1000 kwa mwezi. 
  -> {'profile': {'stock_value_kes': 500000, 'budget_amount_kes': 1000, 'budget_period': 'month', 'primary_risk_concern': 'fire', 'wants_cover_fire': True}, 'language': 'sw', 'intent': 'recommendation'} 

What is microinsurance? 
  -> {'profile': {}, 'language': 'en', 'intent': 'information'} 



## 10. The rules (plain Python, no AI)
These rules check the trader's situation against what the documents say. Every rule result carries the file and page it comes from.

Each product gets one of three statuses:
- `may_qualify`: nothing in the documents rules the trader out, but something still has to be confirmed
- `needs_confirmation`: the documents don't say, so the insurer must confirm
- `does_not_meet`: a requirement stated in the document is not met

A status is never a plain "yes". Documents say premises "should" meet a rule, and the matching database says eligibility must be confirmed in writing.

**Where each rule comes from**
- **TCIP:** the KES 15,000,000 limit, the wall and roof materials and the excluded businesses (banks, forex bureaux, jewellery shops, manufacturing premises) are from `TCIP-BROCHURE.pdf`, p.1. The brochure gives no premium.
- **Britam Biashara:** the guide has no stall-type rule and doesn't list its exclusions; it gives premium *rates*, not a flat premium. The need to confirm open-air stall eligibility in writing is from the matching database (Products P001/P002).
- **Budget:** the conversion (weeks x 52/12, days x 7 x 52/12) follows the workbook's Affordability sheet.

These rules are written by hand from the documents, so if a document changes, the rules must be updated too.

In [25]:
TCIP_SRC = "TCIP-BROCHURE.pdf, p.1"
BRITAM_SRC = "BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.7"
DB_SRC = "ReCover_Microinsurance_Product_Matching_Database.xlsx (Products P001/P002)"

TCIP_MAX_STOCK = 15_000_000   # KES; the brochure's limit for contents and stock
# search word -> readable label. A short word such as "jewel" also catches jewellery, jeweller, jewelry.
TCIP_EXCLUDED = {
    "bank": "banks",
    "forex": "forex bureaux",
    "jewel": "jewellery shops",
    "manufactur": "premises where manufacturing is carried out",
}
STALL_WORDS = {
    "open_air": "an open-air stall",
    "temporary_structure": "a temporary (wooden or mabati) kiosk",
    "permanent_building": "a permanent stone or brick shop",
}
ORDER = {"may_qualify": 0, "needs_confirmation": 1, "does_not_meet": 2}   # best to worst


def worse(a, b):
    """Return the more serious of two statuses, so one failed check is never overwritten by a later pass."""
    return a if ORDER[a] >= ORDER[b] else b


def monthly_budget(profile):
    """Convert the trader's budget to KES per month. Returns None if amount or period is missing."""
    amount, period = profile.get("budget_amount_kes"), profile.get("budget_period")
    if amount is None or period is None:
        return None
    factor = {"month": 1, "week": 52 / 12, "day": 7 * 52 / 12}[period]
    return round(amount * factor)


def check_tcip(profile):
    """Check the trader against the Jubilee TCIP brochure."""
    status, reasons = "may_qualify", []
    stall = profile.get("stall_type")
    stock = profile.get("stock_value_kes")
    business = (profile.get("business_type") or "").lower()

    # Check 1: stock value above the policy limit
    if stock is not None and stock > TCIP_MAX_STOCK:
        status = worse(status, "does_not_meet")
        reasons.append((f"Stock of KES {stock:,} is above the KES {TCIP_MAX_STOCK:,} limit for contents and stock.", TCIP_SRC))

    # Check 2: excluded business type (exact word boundary match so 'power bank' won't trigger 'bank')
    for keyword, label in TCIP_EXCLUDED.items():
         if re.search(r'\b' + re.escape(keyword), business):
             status = worse(status, "does_not_meet")
             reasons.append((f"'{profile['business_type']}' appears to match an excluded business ({label}).", TCIP_SRC))

    # Check 3: the premises. A missing fact gives needs_confirmation, never a guess.
    if stall is None:
        status = worse(status, "needs_confirmation")
        reasons.append(("Premises should have walls of reinforced concrete, concrete blocks, bricks or stone, and a hard roof of tiles, cement or metal sheets. The trader's premises type was not stated.", TCIP_SRC))
    elif stall == "permanent_building":
        # Stays may_qualify: the brochure also requires a hard roof, which the trader hasn't told us about.
        reasons.append(("Stone or brick walls match the wall rule. The roof must be tiles, cement or metal sheets; the roof was not stated.", TCIP_SRC))
    else:
        status = worse(status, "does_not_meet")
        reasons.append((f"Premises should have walls of reinforced concrete, concrete blocks, bricks or stone. {STALL_WORDS[stall].capitalize()} does not match that.", TCIP_SRC))

    # The brochure gives no premium, so say so instead of letting anyone guess one.
    reasons.append(("The brochure does not state a premium.", TCIP_SRC))
    return {"product": "Jubilee Traders Comprehensive Insurance Policy (TCIP)", "status": status, "reasons": reasons}


def check_britam(profile):
    """Check the trader against the Britam Biashara guide. Always needs_confirmation, with the reason spelled out."""
    stall = profile.get("stall_type")
    reasons = [("The guide says the cover applies to all occupations other than the defined exclusions, but it does not list those exclusions or any stall-type rule.", BRITAM_SRC)]
    if stall == "open_air":
        reasons.append(("Open-air stall eligibility must be confirmed in writing by the insurer; do not assume it is covered.", DB_SRC))
    elif stall == "temporary_structure":
        reasons.append(("The guide does not say whether a temporary kiosk qualifies; the insurer must confirm in writing.", BRITAM_SRC))
    else:
        reasons.append(("Eligibility for the trader's premises must be confirmed with the insurer.", DB_SRC))
    reasons.append(("The guide gives premium rates (percentages), not a flat premium, and does not say what amount the rate applies to. A quote is needed.", "BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8"))
    return {"product": "Britam Biashara Cover", "status": "needs_confirmation", "reasons": reasons}


def run_rules(profile):
    """Run every product check, and build one note about the budget. Returns (results, budget_note)."""
    results = [check_tcip(profile), check_britam(profile)]
    monthly = monthly_budget(profile)
    note = None
    if monthly is not None:
        # We cannot say whether a product "fits": neither document gives a flat premium.
        note = (f"The trader's budget is about KES {monthly:,} per month. Neither document gives a flat premium, "
                "so we cannot say whether a product fits the budget until the insurer gives a quote.")
    return results, note

### Test the rules
These checks need no internet or API key. If a rule is changed and breaks, the cell stops with an error instead of quietly giving wrong answers.

In [ ]:
r, _ = run_rules({"stall_type": "permanent_building", "stock_value_kes": 300000})
assert r[0]["status"] == "may_qualify" and r[1]["status"] == "needs_confirmation"

r, _ = run_rules({"stall_type": "temporary_structure", "business_type": "mitumba shoes"})
assert r[0]["status"] == "does_not_meet"

r, note = run_rules({"stall_type": "permanent_building", "business_type": "jewellery",
                     "budget_amount_kes": 100, "budget_period": "week"})
assert r[0]["status"] == "does_not_meet"

assert run_rules({"stall_type": "permanent_building", "stock_value_kes": 20_000_000})[0][0]["status"] == "does_not_meet"
assert run_rules({"stall_type": "open_air"})[0][0]["status"] == "does_not_meet"
assert run_rules({})[0][0]["status"] == "needs_confirmation"      # nothing known -> ask, don't assume
assert monthly_budget({"budget_amount_kes": 20, "budget_period": "day"}) == 607    # matches the workbook (about 606.67)
assert monthly_budget({"budget_amount_kes": 100, "budget_period": "week"}) == 433
assert monthly_budget({}) is None
print("All rule tests passed.")
print(note)

All rule tests passed.
The trader's budget is about KES 433 per month. Neither document gives a flat premium, so we cannot say whether a product fits the budget until the insurer gives a quote.


## 11. The instructions Gemini must follow
This is the standing rulebook sent with every answer. The most important rule is **#1: use only the material we give it.** Without it, Gemini would answer from general knowledge and could invent policy details.

In [ ]:
SYSTEM = """You are 'ReCover', a plain-spoken insurance information assistant for informal traders in Kenya (for example mitumba sellers).

LANGUAGE: Reply in the language the trader used: English, Kiswahili, or a Swahili/English/Sheng mix. Keep it simple, no insurance jargon.

RULES (strict):
1. Use ONLY the EXCERPTS and RULE RESULTS provided. Every number, product name, limit and exclusion must come from them.
2. After each fact, cite its label, like [S1] or [R1]. Never cite a label that was not provided.
3. If the material does not answer part of the question, say clearly that the approved documents do not specify it. Do not guess or use general knowledge.
4. Premiums in the excerpts may be percentage rates. Quote the rate as printed. Do not calculate a premium. Say the exact premium needs a quote from the insurer.
5. If an excerpt looks unclear or contradictory (for example '15% (1.5%)'), say so instead of picking one.
6. Never promise a payout. Mention the conditions and exclusions shown in the material.
7. RULE RESULTS (labelled [R1], [R2]) come from checked rules and are authoritative for eligibility.
   - may_qualify: do NOT say 'yes, you qualify'. Say they may qualify and name what still has to be confirmed.
   - needs_confirmation: say the insurer must confirm eligibility.
   - does_not_meet: say the document's requirement is not met. Do not recommend that product.
8. If a BUDGET NOTE is given, repeat it. Never say a product fits or does not fit the budget.
9. Only mention a product if it is relevant to the question."""

# Shown when we found no relevant evidence. It is fixed text, so no AI call is made and nothing can be invented.
UNAVAILABLE = ("I don't have that information in the approved insurance documents, "
               "so I can't answer it without guessing.")

## 12. The recommendation path
The team's **recommendation service** is a small web program that is already running online. We don't copy its model into this notebook. We send it the trader's details over the internet (an *API call*) and it sends back suggested plans. The address is `RECOMMENDER_URL` in Section 1.

The service needs 11 details about the trader, all required. Most are not in a first message, so the bot asks for them in small groups (2-3 questions per message) instead of all at once.

What this section contains:
- `missing_fields` / `next_question_fields`: what we still need to ask, and which group to ask next.
- `say`: has Gemini write a short message in the trader's language from an English instruction. It is only used for questions and error messages, never for facts.
- `build_payload`: turns our form into the 11 fields the service expects. The monthly budget becomes `wtp_monthly_kes`.
- `call_recommender`: sends the request. A sleeping free server can take about a minute to wake up, so there is a long timeout and one retry.
- `recommendation_facts`: turns the service's reply into labelled facts (`[A1]`, `[A2]`...) for Gemini to explain. Numbers are computed or copied here in Python, not by Gemini.

In [ ]:
import requests

LANG_RULE = {
    "en": "Respond entirely in clear, plain English.",
    "sw": "Jibu kwa Kiswahili rahisi kueleweka. Epuka maneno magumu ya kisheria.",
    "sheng": "Respond in natural Kenyan code-switching (English, Swahili and common Sheng like 'bob', 'kibanda', 'moto').",
}

# The service's reply is cited as [A1], [A2]... with this source line.
SRC_REC = ("ReCover recommendation service (Britam Biashara Lite plan table collected by the team; "
           "prices not yet checked against the brochure)")

# What we still need to ask, in the order we ask it. Each group is asked in ONE message.
QUESTION_GROUPS = [
    ["risks", "stock_value_kes"],
    ["budget", "preferred_premium_frequency"],
    ["monthly_income_kes", "income_receipt_pattern", "income_variability"],
    ["restock_cycle_days", "total_loss_24m_kes"],
]
FIELD_QUESTIONS = {
    "risks": "which business risk they want cover for: fire, theft, or both (recommendations are only available for fire and theft on business stock for now)",
    "stock_value_kes": "roughly how much their stock is worth, in KES",
    "budget": "how much they can pay for insurance, and how often (per day, per week or per month)",
    "preferred_premium_frequency": "how they would prefer to pay: daily, weekly, monthly, or when they restock",
    "monthly_income_kes": "about how much they earn in a month, in KES",
    "income_receipt_pattern": "whether their money comes in daily, weekly or monthly",
    "income_variability": "whether their income is steady, changes sometimes, or changes a lot",
    "restock_cycle_days": "how many days there are between restocking",
    "total_loss_24m_kes": "roughly how much they lost to fire, theft or similar in the last 2 years, in KES (0 if nothing)",
}


def missing_fields(profile):
    """List the details we still need before we can call the recommendation service."""
    missing = []
    if not (profile.get("wants_cover_fire") or profile.get("wants_cover_theft")):
        missing.append("risks")
    if profile.get("stock_value_kes") is None or profile.get("stock_value_kes") <= 0:
        missing.append("stock_value_kes")
    if monthly_budget(profile) is None:             # needs both an amount and a period
        missing.append("budget")
    for key in ["preferred_premium_frequency", "monthly_income_kes", "income_receipt_pattern", "income_variability"]:
        if profile.get(key) is None:
            missing.append(key)
    if profile.get("restock_cycle_days") is None or profile.get("restock_cycle_days") <= 0:
        missing.append("restock_cycle_days")
    if profile.get("total_loss_24m_kes") is None:   # 0 is a valid answer here, so we test for None only
        missing.append("total_loss_24m_kes")
    return missing


def next_question_fields(missing):
    """Pick the first group that still has something missing, and return only its missing fields."""
    for group in QUESTION_GROUPS:
        fields = [f for f in group if f in missing]
        if fields:
            return fields
    return []


def say(instruction, language, fallback, context=""):
    """Have Gemini write ONE short WhatsApp-style message in the trader's language.
    Includes recent context so follow-ups sound natural. Falls back to English text."""
    try:
        prompt = (
            f"{LANG_RULE[language]}\n"
            f"Context so far: {context}\n\n"
            f"Write ONE short, friendly message that does this: {instruction}\n"
            "Do not add facts, prices, product names or promises."
        )
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction="You are 'ReCover', a plain-spoken insurance assistant for Kenyan market traders.",
                temperature=0.3,
            ),
        )
        return resp.text.strip()
    except Exception:
        return fallback

def ask_for_details(fields, language, context=""):
    wanted = "; ".join(FIELD_QUESTIONS[f] for f in fields)
    instruction = ("say you will suggest insurance plans once you know a bit more about their business, and ask for "
                   "ONLY these things: " + wanted)
    fallback = "To suggest a plan I need a few more details. Please tell me " + wanted + "."
    return say(instruction, language, fallback, context=context)

def build_payload(profile):
    """Turn our form into the 11 fields the recommendation service expects."""
    fire = bool(profile.get("wants_cover_fire"))
    theft = bool(profile.get("wants_cover_theft"))
    return {
        "stock_value_kes": float(profile["stock_value_kes"]),
        "total_loss_24m_kes": float(profile["total_loss_24m_kes"]),
        "monthly_income_kes": float(profile["monthly_income_kes"]),
        "wtp_monthly_kes": float(monthly_budget(profile)),      # the trader's budget, converted to KES per month
        "restock_cycle_days": int(profile["restock_cycle_days"]),
        # Ensure the risk passed is strictly accepted by the ML recommender endpoint
        "primary_risk_concern": (
           profile.get("primary_risk_concern")
           if profile.get("primary_risk_concern") in {"fire", "theft"}
           else ("fire" if fire else "theft")
          ),
        "wants_cover_fire": fire,
        "wants_cover_theft": theft,
        "income_variability": profile["income_variability"],
        "preferred_premium_frequency": profile["preferred_premium_frequency"],
        "income_receipt_pattern": profile["income_receipt_pattern"],
    }


def call_recommender(payload):
    """Send the details to the recommendation service. Returns its 'result', or None if it fails."""
    for attempt in (1, 2):   # one retry, because a sleeping server may time out the first time
        try:
            resp = requests.post(RECOMMENDER_URL, json=payload, timeout=RECOMMENDER_TIMEOUT)
            if resp.status_code == 200 and resp.json().get("success"):
                return resp.json()["result"]
            print(f"(recommendation service answered {resp.status_code}: {resp.text[:200]})")
        except Exception as e:
            print(f"(recommendation service attempt {attempt} failed: {e})")
    return None


def recommendation_facts(result, profile):
    """Turn the service's reply into labelled facts [A1], [A2]... plus notes for Gemini.
    Weekly premium is worked out here (annual / 52, the workbook's convention), not by Gemini."""
    labelled, lines, notes = {}, [], []
    priced = above_budget = 0

    for n, r in enumerate(result.get("recommendations", []), start=1):
        label = f"A{n}"
        labelled[label] = SRC_REC
        if "product_name" not in r:    # the service found no plan with a high enough limit for this risk
            lines.append(f"[{label}] No suitable plan found for {r.get('risk_type', 'this risk')}: {r.get('message', 'no detail given')}")
            continue
        annual = float(r["annual_premium_kes"])
        monthly = float(r.get("monthly_premium_equivalent_kes", annual / 12))
        status = r.get("affordability_status", "unknown")
        priced += 1
        if status != "Within stated WTP":
            above_budget += 1
        lines.append(
            f"[{label}] Cover for {r.get('risk_type')}: Britam Biashara Lite plan {r['product_name']}. "
            f"Annual premium KES {annual:,.0f} (about KES {monthly:,.0f} per month, KES {annual / 52:,.0f} per week). "
            f"Cover limit KES {float(r.get('coverage_limit_kes', 0)):,.0f} for stock of KES {float(r.get('stock_value_kes', 0)):,.0f}. "
            f"Affordability as reported: {status} (trader's budget about KES {float(r.get('monthly_wtp_kes', 0)):,.0f} per month)."
        )

    pay = result.get("payment_recommendation") or {}
    if pay:
        label = f"A{len(labelled) + 1}"
        labelled[label] = SRC_REC
        lines.append(f"[{label}] Payment schedule: suggested frequency {pay.get('recommended_frequency')} "
                     f"({pay.get('reason')}). The insurer must confirm that this arrangement is allowed.")

    if priced and above_budget == priced:
        notes.append("EVERY plan listed is above the trader's stated monthly budget. Say this plainly.")
    notes.append("The prices come from the team's Britam Biashara Lite plan table, which has not yet been checked against the brochure. Tell the trader to confirm prices with Britam.")
    notes.append("Whether the trader's premises qualify must be confirmed in writing by Britam.")
    if profile.get("wants_cover_health") or profile.get("wants_cover_accident"):
        notes.append("The trader also mentioned health or accident cover. These plans cover fire and theft on business stock only. Say that other needs are not covered by this recommendation.")
    return labelled, lines, notes


SYSTEM_REC = """You are 'ReCover', a plain-spoken insurance assistant for informal traders in Kenya (for example mitumba sellers).
You explain the plan options below. They come from the team's recommendation service.

RULES (strict):
1. Use ONLY the RECOMMENDATION RESULTS and NOTES. Copy every number exactly as given. Never calculate or change a number. Never mention a plan that is not in the results.
2. After each fact, cite its label, like [A1]. Never cite a label that was not provided.
3. Present the plans as options to explore, not as guaranteed or best-for-sure. Never promise a payout or an approval.
4. State each plan's annual, monthly and weekly premium and cover limit as given, and say whether it is within the trader's budget as reported.
5. Follow every line in NOTES. If a note says every plan is above the budget, say that plainly and do not hide it.
6. Mention the suggested payment schedule, and that the insurer must confirm it.
7. Do not mention clusters, models, scores or how the service works internally.
8. Follow the ACTIVE LANGUAGE instruction. Keep it short and simple, like a WhatsApp message."""

### Test the recommendation helpers
These checks need no internet or API key.

In [ ]:
full = {"wants_cover_fire": True, "stock_value_kes": 500000, "budget_amount_kes": 250, "budget_period": "week",
        "preferred_premium_frequency": "monthly", "monthly_income_kes": 25000, "income_receipt_pattern": "daily",
        "income_variability": "high", "restock_cycle_days": 14, "total_loss_24m_kes": 0}
assert missing_fields(full) == []                                         # 0 losses is a valid answer
assert missing_fields({}) == ["risks", "stock_value_kes", "budget", "preferred_premium_frequency",
                              "monthly_income_kes", "income_receipt_pattern", "income_variability",
                              "restock_cycle_days", "total_loss_24m_kes"]
assert next_question_fields(missing_fields({})) == ["risks", "stock_value_kes"]
assert next_question_fields(["restock_cycle_days"]) == ["restock_cycle_days"]

payload = build_payload(full)
assert payload["wtp_monthly_kes"] == 1083.0                               # 250 per week x 52 / 12
assert payload["primary_risk_concern"] == "fire" and payload["wants_cover_theft"] is False
assert len(payload) == 11

sample_result = {"payment_recommendation": {"recommended_frequency": "weekly", "reason": "Income is highly variable."},
                 "recommendations": [
                     {"risk_type": "fire", "product_name": "Elite", "annual_premium_kes": 6933.0,
                      "monthly_premium_equivalent_kes": 577.75, "coverage_limit_kes": 500000.0,
                      "stock_value_kes": 500000.0, "affordability_status": "Above stated WTP", "monthly_wtp_kes": 400.0},
                     {"risk_type": "burglary", "status": "No suitable product found", "message": "Limits are below the stock value."}]}
labelled, lines, notes = recommendation_facts(sample_result, {})
assert set(labelled) == {"A1", "A2", "A3"} and "KES 133 per week" in lines[0]    # 6933 / 52
assert any("EVERY plan" in n for n in notes)
print("All recommendation helper tests passed.")

All recommendation helper tests passed.


## 13. The chatbot
`ReCover` joins everything. One `ReCover()` object is one conversation, and it **remembers** what the trader has said, so a follow-up like "does that fit my budget?" still knows about the kiosk.

For each message `ask()` does this:
1. Read the message (Section 9): update the trader's profile, note the language and the intent.
2. **Choose the path.** If the trader wants a recommendation, or is answering a question we just asked, take the recommendation path. Otherwise take the information path.

**Information path (`inform`)**
1. Search for relevant pages (Section 8). **If nothing relevant is found, return the fixed message and stop.**
2. Run the rules (Section 10).
3. Give Gemini the rule results and the page excerpts, each with a label (`[R1]`, `[S1]`).

**Recommendation path (`recommend`)**
1. If details are still missing, ask for the next small group of them and stop.
2. Otherwise call the recommendation service, turn its reply into labelled facts (`[A1]`...) and let Gemini explain them.

Either way, the sources listed at the end are only the labels the answer actually used.

In [ ]:
def collect_sources(text, labelled):
    """Find the labels the answer used. Gemini sometimes writes several in one bracket, like [S1, R1],
    so we read everything inside each [...] and pick out every S#, R# or A# label."""
    cited = []
    for group in re.findall(r"\[([^\]]+)\]", text):
        for label in re.findall(r"[SRA]\d+", group):
            if label not in cited:
                cited.append(label)
    return [f"[{c}] {labelled[c]}" for c in cited if c in labelled]


class ReCover:
    def __init__(self):
        self.profile = {}      # what the trader has told us so far
        self.history = []      # the trader's earlier messages (not the bot's long answers)
        self.language = "en"   # the language of the trader's latest message
        self.awaiting = False  # True after we asked the trader for more details

    def search_text(self, message):
        """Short follow-ups ('does that fit my budget?') are too vague to search alone,
        so with fewer than 8 words we attach the previous message."""
        if self.history and len(message.split()) < 8:
            return f"{self.history[-1]} {message}"
        return message

    def ask(self, message):
        meta = interpret_message(message)
        before = dict(self.profile)
        self.profile.update(meta["profile"])   # a new value (for example a corrected stock figure) replaces the old one
        self.language = meta["language"]
        gained = [k for k, v in self.profile.items() if before.get(k) != v]   # facts that are new or changed this turn

        # A trader answering our question usually sends just a number, which Gemini may label "information".
        # So if we were waiting for details and this message gave us some, stay on the recommendation path.
        if meta["intent"] == "recommendation" or (self.awaiting and gained):
            text, sources = self.recommend(message)
        else:
            text, sources = self.inform(message)
        self.history.append(message)
        return text, sources

    # ---------- recommendation path ----------
    def recommend(self, message):
        missing = missing_fields(self.profile)
        if missing:
            self.awaiting = True
            recent_context = " | ".join(self.history[-2:]) if self.history else "New conversation"
            return ask_for_details(next_question_fields(missing), self.language, context=recent_context), []

        self.awaiting = False
        result = call_recommender(build_payload(self.profile))
        if result is None:
            return say("tell the trader the recommendation service is not responding right now and ask them to try again in a few minutes",
                       self.language, "Sorry, the recommendation service is not responding right now. Please try again in a few minutes."), []

        labelled, lines, notes = recommendation_facts(result, self.profile)
        prompt = (
            f"ACTIVE LANGUAGE: {LANG_RULE[self.language]}\n\n"
            f"WHAT THE TRADER HAS TOLD US: {self.profile}\n\n"
            "RECOMMENDATION RESULTS:\n" + "\n".join(lines) + "\n\n"
            "NOTES:\n" + "\n".join(f"- {n}" for n in notes) + "\n\n"
            f"TRADER MESSAGE: {message}"
        )
        resp = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_REC, temperature=0.2),
        )
        return resp.text, collect_sources(resp.text, labelled)

    # ---------- information path ----------
    def inform(self, message):
        # 1. Find the evidence. No evidence = fixed message, no AI answer.
        hits = retrieve(self.search_text(message))
        if not hits:
            return UNAVAILABLE, []

        # 2. Run the rules, but only if the trader has told us something.
        results, budget_note = run_rules(self.profile) if self.profile else ([], None)

        # 3. Build the labelled material for Gemini.
        labelled = {}   # label -> readable source, used for the source list at the end
        rule_lines, excerpt_lines = [], []
        for i, r in enumerate(results, start=1):
            sources = list(dict.fromkeys(src for _, src in r["reasons"]))   # unique sources, in order
            labelled[f"R{i}"] = f"rules check: {r['product']} (based on {'; '.join(sources)})"
            lines = "\n".join(f"  - {text} ({src})" for text, src in r["reasons"])
            rule_lines.append(f"[R{i}] {r['product']}: status = {r['status']}\n{lines}")
        for i, h in enumerate(hits, start=1):
            labelled[f"S{i}"] = f"{h['source']}, p.{h['page']}"
            excerpt_lines.append(f"[S{i}] ({h['source']}, p.{h['page']})\n{h['text']}")

        prompt = (
            f"ACTIVE LANGUAGE: {LANG_RULE[self.language]}\n\n"
            f"WHAT THE TRADER HAS TOLD US SO FAR: {self.profile or 'nothing yet'}\n\n"
            f"TRADER'S EARLIER MESSAGES: {self.history[-3:] or 'none'}\n\n"
            "RULE RESULTS:\n" + ("\n\n".join(rule_lines) or "(none)") + "\n\n"
            f"BUDGET NOTE: {budget_note or '(none)'}\n\n"
            "EXCERPTS:\n" + "\n\n".join(excerpt_lines) + "\n\n"
            f"TRADER QUESTION: {message}"
        )
        resp = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.2),   # low temperature = steady, factual
        )
        return resp.text, collect_sources(resp.text, labelled)


def chat(bot, message):
    """Send one message and print the answer with its sources."""
    print(f"Trader: {message}\n")
    text, sources = bot.ask(message)
    print(f"ReCover: {text}\n")
    if sources:
        print("Sources:\n  " + "\n  ".join(sources))
    print("\n" + "=" * 70 + "\n")

## 14. Try it
**First, check the recommendation service is reachable.** If it was asleep, this can take about a minute.

Then three conversations:
- **Recommendation:** a trader gives details over several messages. Each reply either asks for the next few details or, once everything is known, explains the plans. If every plan is above the trader's budget, the answer must say so.
- **Information:** the kiosk and stone-shop questions from before.
- **Switching:** after a recommendation, the same trader asks an information question.

In [ ]:
sample_trader = {"stock_value_kes": 150000, "total_loss_24m_kes": 50000, "monthly_income_kes": 25000,
                 "wtp_monthly_kes": 500, "restock_cycle_days": 14, "primary_risk_concern": "fire",
                 "wants_cover_fire": True, "wants_cover_theft": False, "income_variability": "high",
                 "preferred_premium_frequency": "weekly", "income_receipt_pattern": "daily"}
res = call_recommender(sample_trader)
print("Recommendation service reachable:", res is not None)
if res:
    print("Reply contains:", list(res.keys()))

Recommendation service reachable: True
Reply contains: ['cluster', 'cluster_preferred_products', 'payment_recommendation', 'recommendations']


In [ ]:
rec_bot = ReCover()
chat(rec_bot, "Habari, nataka ushauri wa bima ya moto kwa stock yangu ya mitumba. Thamani yake ni 500,000 bob.")
chat(rec_bot, "Naweza lipa 1000 kwa mwezi, napendelea kulipa kila mwezi.")
chat(rec_bot, "Napata kama 25,000 kwa mwezi, pesa inaingia kila siku, na inabadilika sana.")
chat(rec_bot, "Naongeza stock kila siku 14 na sikupoteza kitu miaka miwili iliyopita.")
# The same trader now asks an information question (this should use the document path, not ask for details):
chat(rec_bot, "What does the fire cover include?")

ReCover is not defined. Please run the cell under Section 13 (cell_id: a86913d3) first to initialize the chatbot, then run this cell again!


In [ ]:
bot = ReCover()
chat(bot, "Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto.")
chat(bot, "My budget ni around 500 bob a month. Je, hiyo bima inatoshea bajeti yangu?")

bot2 = ReCover()
chat(bot2, "Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?")
chat(bot2, "Who will win the football match on Sunday?")

bot3 = ReCover()
chat(bot3, "What is the premium rate for theft cover?")

Trader: Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto.

ReCover: Sasa! Naitwa ReCover. Nikishajua kiasi kuhusu kibanda yako, nitakushow plans za insurance zitakazofaa biashara yako. Kwanza, unaona unaweza lipa bob ngapi na ni per day, per week, ama per month? Alafu pia, ungependa kulipa aje: daily, weekly, monthly, ama kila time unarestock?



Trader: My budget ni around 500 bob a month. Je, hiyo bima inatoshea bajeti yangu?

ReCover: Sasa! Nikishajua kiasi kuhusu biashara yako, nitakutafutia plan fiti. Kwa sasa, ungependelea kulipa aje: daily, weekly, monthly, ama kila time unarestock kibanda?



Trader: Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?

ReCover: Huenda ukakubaliwa (unaweza kufuzu) kupata bima ya Jubilee Traders Comprehensive Insurance Policy (TCIP) [R1].

Kuta zako za mawe zinatimiza sharti la kuta za jengo [R1, S2]. Hata hivyo, bado hujakubaliwa moja kwa moja kwa sababu kuna mambo kadhaa yanayofaa kuthi

## 15. Test with the team's own trader questions
Your workbook's `Trader_Questions` sheet lists the 21 questions the chatbot must answer. This cell runs them (each in a fresh conversation) and saves the answers to `data/test_results.csv`, so a teammate can read through and mark which ones are right.

`QUESTION_LIMIT = 5` runs only the first 5, to save time and API calls. Set it to `None` to run all 21.

**Read the answers yourself.** The chatbot passing its own tests proves nothing. A person has to check each answer against the documents, and have a native Swahili speaker review the Swahili.

In [ ]:
QUESTION_LIMIT = 5   # set to None to run all of them


def load_trader_questions():
    """Read the Trader_Questions sheet. The real table starts a few rows down, so we find the header row by looking for 'ID'."""
    workbook = next(REPO_ROOT.rglob("Microinsurance_Data_Collection_Workbook.xlsx"), None)
    if workbook is None:
        raise FileNotFoundError("Microinsurance_Data_Collection_Workbook.xlsx was not found in the repo.")
    raw = pd.read_excel(workbook, sheet_name="Trader_Questions", header=None)
    header_row = next(i for i, row in raw.iterrows() if str(row.iloc[0]).strip() == "ID")
    df = pd.read_excel(workbook, sheet_name="Trader_Questions", header=header_row)
    df = df[df["ID"].astype(str).str.match(r"Q\d+")]       # keep only the real question rows
    df = df.copy()
    # Q12 contains a placeholder; fill in an example amount so it can be asked
    df["Question (English)"] = df["Question (English)"].str.replace("[amount]", "150", regex=False)
    return df


questions = load_trader_questions()
if QUESTION_LIMIT:
    questions = questions.head(QUESTION_LIMIT)

rows = []
for _, q in questions.iterrows():
    try:
        text, sources = ReCover().ask(q["Question (English)"])   # a new ReCover() per question = no memory carried over
    except Exception as e:
        text, sources = f"ERROR: {e}", []
    rows.append({"id": q["ID"], "category": q["Category"], "question": q["Question (English)"],
                 "answer": text, "sources": " | ".join(sources)})
    print(f"{q['ID']}: {q['Question (English)']}\n  -> {text[:200]}...\n")

results_df = pd.DataFrame(rows)
results_df.to_csv("data/test_results.csv", index=False)
print("Saved data/test_results.csv")

Q01: What is microinsurance?
  -> Based on the official documents:

* **Designed for low-income earners:** Microinsurance is a financial service tailored to be appropriate for low-income households and individuals [S2, S3]. 
* **Fully...

Q02: Is insurance for someone like me, who sells mitumba?
  -> Yes, insurance options exist for traders, but your eligibility must be confirmed with the insurer depending on your stall or shop set-up [R1, R2]. 

Here are two covers from the approved documents:

#...

Q03: Is this company licensed and safe?
  -> The approved documents do not specify whether the company you are asking about is licensed or safe, as they do not list individual licensed companies or their status. 

To help you check:
* Under the ...

Q04: What happens if I get sick and cannot work?
  -> Based on the approved documents, here is what happens regarding medical expenses and being unable to work:

**1. Lost Income or Daily Wages**
* The approved documents do not specify any cov

## 16. Limits of this version
**Information side**
- **Rules cover two business products only** (Jubilee TCIP and Britam Biashara). The matching database lists 15 products, but it has no eligibility rules or premiums for the others, so other questions are answered only from retrieved text, with no rule checks.
- **Britam Biashara (network guide) has no flat premium.** It gives percentage rates only, so the bot tells the trader to get a quote. This is different from the Biashara Lite plans used by the recommendation service.
- **Rules are written by hand** from the documents. If a document changes, update Section 10.
- **The Britam "All Risks" rate** is printed as `15% (1.5%)` in the guide, which is unclear. The bot is told to say so.
- **OCR can make mistakes.** Check the transcriptions of scanned PDFs (Section 4) against the originals.

**Recommendation side**
- **The plan prices come from the team's Britam Biashara Lite plan table**, which has not been checked against the brochure. The brochure is not yet in the document library, so the information path cannot cite it.
- **The recommendation model was built on synthetic trader data.** Its trader groups ("clusters") are not evidence about real traders, and the results should be treated as examples until it is tested with real traders.
- **How plans are chosen:** the service takes the plans its model prefers for the trader's group, keeps those whose limit covers the stock, and orders them by smallest coverage gap. Affordability is shown but is not used to choose or order plans, so the assistant reports clearly when plans are above the trader's budget.
- **Fire and theft only,** on business stock. Health, accident and weather cover are not recommended.
- **Nine details are needed** (risk, stock, budget, payment preference, income, how income arrives, how steady it is, restock cycle, past losses). The bot asks for them in groups over several messages. The accepted words for some of them (Section 9) must match the dataset the model was trained on.
- **Needs the service online.** A sleeping free server can take about a minute to answer the first time.

**Both**
- **The fixed "I don't have that information" message is English only.**
- **Swahili and Sheng answers need review** by a native speaker before real traders see them.
- **Not on WhatsApp yet.** This notebook is the brain. A WhatsApp connection (a webhook that receives messages and sends the replies, with one `ReCover()` per phone number) is the next piece.
- This tool gives information and suggestions, not insurance advice. Eligibility and premiums must always be confirmed with the insurer.